#### Imports

In [1]:
import queue
import logging
import pandas as pd
from googlenewsdecoder import gnewsdecoder
from newspaper import Article, Config
from newspaper.article import ArticleException
from concurrent.futures import ThreadPoolExecutor, as_completed
from selenium import webdriver
from selenium.webdriver import ChromeOptions
from tqdm import tqdm
tqdm.pandas()

#### Setup Scraping Environment

- NDTV and Telegraph require special handling using selinium, as they block traditional scraping methods
- Selinium opens a browser window and fetches a fully loaded HTML file for our processing

In [2]:
logging.basicConfig(
    filename='runs_budget.log',
    filemode='a',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [3]:
options = ChromeOptions()
options.page_load_strategy = 'eager'
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-features=IsolateOrigins,site-per-process")
options.add_argument("--disable-extensions")
options.add_argument("--blink-settings=imagesEnabled=false")

options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

THREAD_EXEC = ThreadPoolExecutor(max_workers=20)

# make a pre warmed pool of 10 drivers (chromium heads)
DRIVER_POOL = queue.Queue()
for _ in range(10):
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(20) 
    DRIVER_POOL.put(driver)

def get_selenium_html(url):
    driver = DRIVER_POOL.get(timeout=10)
    try:
        driver.get(url)
        article_html = driver.page_source
        return article_html 
    except Exception:
        try:
            driver.quit()
        except:
            pass
        driver = webdriver.Chrome(options=options)
        driver.set_page_load_timeout(20)
        return None
    finally:
        driver.delete_all_cookies() # denavigate from opened news links, may guzzle RAM
        driver.get("about:blank")
        DRIVER_POOL.put(driver)

config = Config()
config.browser_user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124  Safari/537.36'
config.request_timeout = 20

In [5]:
def parse_url(url) -> str | None:
    try: # first attempt
        article = Article(url)
        article.download()
        article.parse()

        text = " ".join(article.text.split()[:200])
        del article

        if text == '':
            raise LookupError(f"\n[INFO] First Attempt: Unable to extact article text for [{url}]. Retrying...") # if fail, go into second try

        return text
    
    except (ArticleException, LookupError) as e1:
        try: # second attempt (identical)
            if str(e1).find(' 403 ') != -1 or isinstance(e1, LookupError):
                article = Article(url)
                article.download()
                article.parse()

                text = " ".join(article.text.split()[:200])
                del article

                if text == '':
                    raise LookupError(f"\n[INFO] Second Attempt: Unable to extact article text for [{url}]. Retrying...") # if fail, go into third try
                else:
                    logging.info(f'Retry succeeded for [{url}]!')
                    
                return text
            
            else:
                logging.error(str(e1))
                return ''
            
        except (ArticleException, LookupError) as e2:
            try: # 3rd attempt (with config)
                if str(e2).find(' 403 ') != -1 or isinstance(e2, LookupError):
                    article = Article(url, config=config)
                    article.download()
                    article.parse()
                    text = " ".join(article.text.split()[:200])
                    del article

                    if text == '':
                        raise LookupError(f"\n[INFO] Third Attempt: Unable to extact article text for [{url}]. Retryig...") # if fail, final try
                    else:
                        logging.info(f'Retry succeeded for [{url}]!')
                    
                    return text

                else:
                    logging.error(str(e2))
                    return ''

            except (ArticleException, LookupError) as e3:
                # final attempt (config and selenium)
                if str(e3).find(' 403 ') != -1 or isinstance(e3, LookupError):
                    article = Article(url, config=config)
                    article.download(input_html = get_selenium_html(url))
                    article.parse()
                    text = " ".join(article.text.split()[:200])
                    del article

                    if text == '':
                        logging.error(f"Final Attempt: Unable to extact article text for [{url}]") # if it still fails, can't circumvent.
                        #print(e3)
                    else:
                        logging.info(f'Retry succeeded for [{url}]!')
                    
                    return text

                else:
                    logging.error(str(e3))
                    return ''

def parse_url_safe(url):
    try:
        return parse_url(url)
    except Exception as e:
        logging.error(f"Unhandled exception parsing [{url}]: {e}")
        return ''

def decode_url(url:str):
    return gnewsdecoder(url)['decoded_url']

def parse_stories_parallel(stories, max_threads=15):
    future_to_story = {
        THREAD_EXEC.submit(parse_url, story['url']): story 
        for story in stories
    }
    
    for future in as_completed(future_to_story):
        story = future_to_story[future]
        try:
            text = future.result()
            story['text'] = text
            
        except Exception as e:
            print(f"Unexpected failure processing [{story['url']}]: {e}")
            story['text'] = ''

    return stories

def decode_urls_parallel(stories:list[dict[str|str]], max_threads=15):
    future_to_story = {
        THREAD_EXEC.submit(decode_url, story['url']): story 
        for story in stories
    }
    
    for future in as_completed(future_to_story):
        story = future_to_story[future]
        try:
            url = future.result()
            story['url'] = url
            
        except Exception as e:
            print(f"Unexpected failure processing [{story['url']}]: {e}")
            story['url'] = ''

    return stories

def decoding_block(request_stories: list[dict[str,str]]):
    stories = decode_urls_parallel(request_stories)
    return [story for story in stories if story['url'] != '']

def scraping_block(stories: list[dict[str,str]]):
    stories = parse_stories_parallel(stories)
    return [story for story in stories if story['text'] != '']

def shutdown_selenium_pool():
    while not DRIVER_POOL.empty():
        driver = DRIVER_POOL.get()
        driver.quit()
    print("INFO:     Headerless browsers quit.")

def parallel_apply(series: pd.Series) -> pd.Series:
    results = pd.Series(index=series.index, dtype=object)
    future_to_idx = {THREAD_EXEC.submit(parse_url_safe, val): idx for idx, val in series.items()}

    for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="Processing..."):
        idx = future_to_idx[future]
        try:
            results[idx] = future.result()
        except Exception as e:
            logging.error(f"Unhandled exception for [{series.at[idx]}]: {e}")
            results[idx] = ''
    return results

In [6]:
parse_url('https://www.ndtv.com/india-news/j-k-important-part-of-india-us-ambassador-sergio-gor-in-srinagar-11930634')

''

#### Load CSV

In [7]:
event = 'budget'

In [8]:
df = pd.read_csv(f'../{event}_reingest.csv')

df

,Unnamed: 0,id,indexed_date,language,media_name,media_url,publish_date,title,url
0,0,0ca24f8e5e4ee0dfa7a52c1c64f5d4d4afa6ba5abcdbf0...,2026-03-21 14:22:33.811807+00:00,en,thehindu.com,thehindu.com,2016-03-02,‘Launch of an insurrection against the Constit...,https://frontline.thehindu.com/cover-story/lau...
1,1,826ded8dcbfbb979d3dfbc67629d38120447c068cf7a34...,2025-09-20 01:13:19.155781+00:00,en,bbc.co.uk,bbc.co.uk,2011-03-02,India: The next university superpower?,http://www.bbc.co.uk/go/rss/int/news/-/news/bu...
2,2,41ea31bb141415c2d159188095e69a486332cb7eff88e4...,2025-09-16 18:46:49.587199+00:00,en,indiatimes.com,indiatimes.com,2013-03-21,Maruti Swift hits discount road in sales push ...,http://economictimes.indiatimes.com/news/news-...
3,3,fb80a7f207acdf0a12d2db0c495877f922f2f26d414b65...,2025-09-16 18:32:23.715725+00:00,en,thehindu.com,thehindu.com,2013-03-22,GST has a good probability in tax horizon: Par...,http://www.thehindu.com/business/Industry/gst-...
4,4,339124be2f7fd5a2748a6ed21fec5eff0ee3bdbfd32ba3...,2025-09-16 17:51:40.927071+00:00,en,dnaindia.com,dnaindia.com,2013-01-01,2013: New hopes and a fresh beginning...,http://www.dnaindia.com/speakup/1784296/report...
...,...,...,...,...,...,...,...,...,...
29260,29260,646292395e078f075961bb81006a3b38cda417118efc9e...,2023-11-15 00:23:31+00:00,en,india.com,india.com,2023-11-13,PM Modi To Launch PM PVTG (Particularly Vulner...,https://www.india.com/news/india/pm-modi-to-la...
29261,29261,16c7f0238069bef4718107bc6cd6e5d3b47366b027f104...,2023-11-13 01:44:12+00:00,en,ndtv.com,ndtv.com,2023-11-11,National Education Day: How Is India Leading I...,https://www.ndtv.com/education/national-educat...
29262,29262,2d7f277cf182f2e2c9c41a7d3516d6ec6f22205cf225f5...,2023-11-13 00:09:45+00:00,en,indiatimes.com,indiatimes.com,2023-02-07,GST on digital services: Budget 2023-24 broade...,https://economictimes.indiatimes.com/small-biz...
29263,29263,7753d0a284ba8f43b17bfa822a5a09068267fdf22f7ca9...,2023-11-12 00:15:06+00:00,en,thehindu.com,thehindu.com,2023-03-03,Programme to upskill construction workers,https://www.thehindu.com/life-and-style/homes-...


#### Extract Text and Various Fields

In [9]:
newspaper3k_df = parallel_apply(df['url'])

Processing...: 100%|██████████| 29265/29265 [51:50<00:00,  9.41it/s]  


In [10]:
newspaper3k_df = pd.DataFrame(data = newspaper3k_df.tolist(), columns = ['Text'], index=df.index)

newspaper3k_df.head()

,Text
0,Several Left party leaders have studied in JNU...
1,
2,It seems like you're already an ETPrime member...
3,Advisor to Union Finance Minister Parthasarath...
4,


In [11]:
df = df.join(newspaper3k_df)

df.head()

,Unnamed: 0,id,indexed_date,language,media_name,media_url,publish_date,title,url,Text
0,0,0ca24f8e5e4ee0dfa7a52c1c64f5d4d4afa6ba5abcdbf0...,2026-03-21 14:22:33.811807+00:00,en,thehindu.com,thehindu.com,2016-03-02,‘Launch of an insurrection against the Constit...,https://frontline.thehindu.com/cover-story/lau...,Several Left party leaders have studied in JNU...
1,1,826ded8dcbfbb979d3dfbc67629d38120447c068cf7a34...,2025-09-20 01:13:19.155781+00:00,en,bbc.co.uk,bbc.co.uk,2011-03-02,India: The next university superpower?,http://www.bbc.co.uk/go/rss/int/news/-/news/bu...,
2,2,41ea31bb141415c2d159188095e69a486332cb7eff88e4...,2025-09-16 18:46:49.587199+00:00,en,indiatimes.com,indiatimes.com,2013-03-21,Maruti Swift hits discount road in sales push ...,http://economictimes.indiatimes.com/news/news-...,It seems like you're already an ETPrime member...
3,3,fb80a7f207acdf0a12d2db0c495877f922f2f26d414b65...,2025-09-16 18:32:23.715725+00:00,en,thehindu.com,thehindu.com,2013-03-22,GST has a good probability in tax horizon: Par...,http://www.thehindu.com/business/Industry/gst-...,Advisor to Union Finance Minister Parthasarath...
4,4,339124be2f7fd5a2748a6ed21fec5eff0ee3bdbfd32ba3...,2025-09-16 17:51:40.927071+00:00,en,dnaindia.com,dnaindia.com,2013-01-01,2013: New hopes and a fresh beginning...,http://www.dnaindia.com/speakup/1784296/report...,


In [12]:
shutdown_selenium_pool()

INFO:     Headerless browsers quit.


In [13]:
df.to_csv(f'Extracted/{event}_reingest.csv', index=False)